In [1]:
# Cell 1: Imports & Setup
import pandas as pd
from pathlib import Path
import json, gc
import numpy as np
import polars as pl
import spatial_split as ss

data_base_dir = Path('data/regionwise_yearwise_samples')
output_dir = Path('data/feature_selection_artifacts')
output_dir.mkdir(parents=True, exist_ok=True)

import os
# Regions to process: set here, or via the REGIONS environment variable (comma separated)
# TARGET_REGIONS = os.environ.get('REGIONS', 'HIM').split(',')
TARGET_REGIONS = ['HIM', 'PEN', 'DES', 'IGP', 'WGR', 'ICR', 'NEI']

# Predictor exclusions: metadata, target, and null-heavy composite bands (lat/lon are kept as predictors)
cols_to_drop = [
    'class', 'system:index', 'area_m2', 'area_proportion', 'usable_count',
    '.geo', 'q1_count', 'q2_count', 'q3_count', 'q4_count', 'quarters_present',
    'geometry', 'ndvi_q1_median', 'ndvi_q2_median', 'ndvi_q3_median',
    'ndvi_q4_median', 'tir_count', 'tir_mad', 'tir_median', 'snow_count',
    'year', 'mosaic_version', 'split',
    'BareEarth', 'StableWater', 'TransientWater',   # ancillary mask columns, not predictors
]
pixel_quality_condition = ss.QUALITY          # usable_count >= 5 & quarters_present >= 3
class_remap_expr = ss._remap()                # Level 2 -> Level 1 class remap, same as every other step

In [2]:
# Cell 2: Drop collinear features (|r| > 0.80), using train-split pixels of the usable years
THRESHOLD = 0.80
for target_region in TARGET_REGIONS:
    print(f"\n{'='*50}\nProcessing region: {target_region}\n{'='*50}")
    years = ss.study_years(target_region)
    assert years, f'run 03_split_and_years.ipynb for {target_region} first'

    lazy_ds = ss.scan_years(target_region, years)
    feature_cols = [c for c in lazy_ds.collect_schema().names() if c not in cols_to_drop]

    df_collected = (
        ss.add_split(lazy_ds.with_columns(class_remap_expr), target_region)
        .filter(pixel_quality_condition & (pl.col('split') == 'train'))
        .select(feature_cols + ['class'])
        .collect()
    )
    print(f"[{target_region}] Train-split gated rows: {df_collected.height:,}")
    df_sample = df_collected.sample(n=min(50000, df_collected.height), seed=42)
    del df_collected; gc.collect()

    X = df_sample.select(feature_cols).to_numpy().astype(np.float32)
    y = df_sample['class'].to_numpy().astype(np.float32)
    del df_sample; gc.collect()

    corr_df = pd.DataFrame(np.abs(np.corrcoef(X, rowvar=False)), index=feature_cols, columns=feature_cols)
    y_n = (y - y.mean()) / y.std()
    X_n = (X - X.mean(axis=0)) / X.std(axis=0)
    target_corr = pd.Series(np.abs(np.mean(X_n * y_n[:, None], axis=0)), index=feature_cols)

    upper = corr_df.where(np.triu(np.ones(corr_df.shape), k=1).astype(bool))
    to_drop = set()
    for col in upper.columns:
        for feat in upper.index[upper[col] > THRESHOLD].tolist():
            to_drop.add(feat if target_corr[feat] < target_corr[col] else col)
    uncorrelated = [c for c in feature_cols if c not in to_drop]

    out = output_dir / f"{target_region}_uncorrelated_features.json"
    json.dump({"region": target_region, "years": years, "corr_threshold": THRESHOLD,
               "total_raw_features": len(feature_cols), "uncorrelated_features": uncorrelated,
               "dropped_features": sorted(to_drop)}, open(out, "w"), indent=2)
    print(f"[{target_region}] raw {len(feature_cols)} -> dropped {len(to_drop)} -> kept {len(uncorrelated)}  ({out})")
    del X, y, corr_df, X_n, y_n, upper; gc.collect()


Processing region: HIM
[HIM] Train-split gated rows: 1,147,976
[HIM] raw 103 -> dropped 72 -> kept 31  (data/feature_selection_artifacts/HIM_uncorrelated_features.json)

Processing region: PEN
[PEN] Train-split gated rows: 978,187
[PEN] raw 103 -> dropped 67 -> kept 36  (data/feature_selection_artifacts/PEN_uncorrelated_features.json)

Processing region: DES
[DES] Train-split gated rows: 3,065,162
[DES] raw 103 -> dropped 71 -> kept 32  (data/feature_selection_artifacts/DES_uncorrelated_features.json)

Processing region: IGP
[IGP] Train-split gated rows: 2,002,375
[IGP] raw 103 -> dropped 68 -> kept 35  (data/feature_selection_artifacts/IGP_uncorrelated_features.json)

Processing region: WGR
[WGR] Train-split gated rows: 982,073
[WGR] raw 103 -> dropped 74 -> kept 29  (data/feature_selection_artifacts/WGR_uncorrelated_features.json)

Processing region: ICR
[ICR] Train-split gated rows: 1,710,508
[ICR] raw 103 -> dropped 66 -> kept 37  (data/feature_selection_artifacts/ICR_uncorrelated